# Запуск пайплайна отбора факторов

Ноутбук формирует рабочие YAML и запускает подготовку выборок, прямой selector или stability selection. При `RUN_SAMPLE_PREPARATION = False` существующие `split_parts` и sample-конфиг не изменяются.

**Порядок:** настройте первую кодовую ячейку, затем выполняйте остальные сверху вниз. Токен внутреннего PyPI вводится скрыто.

In [ ]:
# 1. CONFIG
import os
import json
import subprocess
import sys
from getpass import getpass
from pathlib import Path
from urllib.parse import quote


# =============================================================================
# CONFIG: EDIT ONLY THIS BLOCK
# =============================================================================
PROJECT_DIR = Path("/home/datalab/nfs/Пробник").resolve()
SOURCE_PATH = Path("/path/to/modeling_population").resolve()
SOURCE_FORMAT = "parquet"

TARGET = "del30_4"
POSITIVE_LABEL = 1
ID_COLUMNS = [
    "model_appl_num",
    "model_appl_dt",
    "model_cust_epk_sid",
    "appl_num",
    "appl_global_sid",
    "bki_reqst_sid",
]
SAMPLING_KEY_COLUMNS = ["model_appl_num"]
LEAKAGE_KEY_COLUMNS = ["model_cust_epk_sid"]
BOOTSTRAP_KEY_COLUMNS = ["model_cust_epk_sid"]
CATEGORICAL_FEATURES: list[str] = []
CATEGORICAL_NULL_STRATEGY = "fill"  # fill or error
CATEGORICAL_NULL_VALUE = "__MISSING__"
EXCLUDED_FEATURES: list[str] = []
REQUIRED_FEATURES: list[str] = []

SPLIT_MODE = "auto"
TRAIN_FRACTION = 0.55
VALID_FRACTION = 0.15
OOS_FRACTION = 0.15
OOT_FRACTION = 0.15
TEST_FRACTION = 0.0
TIME_COLUMN = "model_appl_dt"
GROUP_COLUMNS: list[str] = []
OVERWRITE_SPLITS = False
N_FOLDS = 3

PRIMARY_METRIC = "average_precision"
MIN_FEATURES = 10
MAX_FEATURES = 100
MAX_GAP = 0.15
BOOTSTRAP_REPEATS = 1000
MIN_RECALL = 0.50
MIN_PRECISION = 0.20

TASK_TYPE = "GPU"
CPU_CORES = 8
RAM_GB = 128.0
GPU_DEVICES = ["0", "1"]
GPU_MEMORY_BY_DEVICE_GB = {"0": 32.0, "1": 32.0}
GPU_RAM_PART = 0.70
GPU_RESERVE_MEMORY_GB = 2.0
GPU_MAX_MEMORY_UTILIZATION = 0.90
CALIBRATION_SAFETY_FACTOR = 1.20

INSTALL_REQUIREMENTS = True
FORCE_REINSTALL = False
INSTALL_RAY = False
RUN_SAMPLE_PREPARATION = False
RUN_MODE = "stability"  # direct, stability, none
RUN_FINAL_SELECTOR = True

PYPI_HOST = "sberosc.ca.sbrf.ru"
PYPI_URL_TEMPLATE = f"https://token:{{token}}@{PYPI_HOST}/repo/pypi/simple"


# =============================================================================
# PATHS
# =============================================================================
REQUIREMENTS = PROJECT_DIR / "requirements.txt"
SAMPLE_TEMPLATE = PROJECT_DIR / "config.sample-preprocessing.example.yaml"
INTERACTION_TEMPLATE = PROJECT_DIR / "config.interaction-selector.example.yaml"
STABILITY_TEMPLATE = PROJECT_DIR / "config.stability-selection.example.yaml"
SAMPLE_CONFIG = PROJECT_DIR / "config.sample-preprocessing.yaml"
INTERACTION_CONFIG = PROJECT_DIR / "config.interaction-selector.yaml"
STABILITY_CONFIG = PROJECT_DIR / "config.stability-selection.yaml"
PREPARE_SCRIPT = PROJECT_DIR / "prepare_selector_samples.py"
SELECTOR_SCRIPT = PROJECT_DIR / "interaction_subset_selector.py"
STABILITY_RUNNER = PROJECT_DIR / "run_stability_selection.py"
SPLIT_DIR = PROJECT_DIR / "split_parts"
SELECTOR_DATA_CONFIG = SPLIT_DIR / "selector_data.yaml"
SPLIT_MANIFEST = SPLIT_DIR / "split_manifest.json"
FOLDS_MANIFEST = SPLIT_DIR / "folds_manifest.json"
DIRECT_OUTPUT = PROJECT_DIR / "interaction_selection_results_v10"
STABILITY_OUTPUT = PROJECT_DIR / "stability_selection"


In [ ]:
# 2. СЛУЖЕБНЫЕ ФУНКЦИИ
def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Не найден файл: {path}")


def run(command: list[str], env: dict[str, str] | None = None) -> None:
    print("\n" + "=" * 80, "ЗАПУСК:", " ".join(command), "=" * 80, sep="\n")
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def install_packages() -> None:
    require_file(REQUIREMENTS)
    token = os.environ.get("OSC_TOKEN") or getpass(
        "Введите токен внутреннего PyPI: "
    ).strip()
    if not token:
        raise ValueError("Токен внутреннего PyPI не указан")
    env = os.environ.copy()
    env["PIP_INDEX_URL"] = PYPI_URL_TEMPLATE.format(token=quote(token, safe=""))
    env["PIP_TRUSTED_HOST"] = PYPI_HOST
    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    command = [sys.executable, "-m", "pip", "install", "--no-cache-dir"]
    if FORCE_REINSTALL:
        command.append("--force-reinstall")
    command.extend(["--trusted-host", PYPI_HOST, "-r", str(REQUIREMENTS)])
    run(command, env)
    if INSTALL_RAY:
        run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-cache-dir",
                "--trusted-host",
                PYPI_HOST,
                "ray[default]>=2.40,<3",
            ],
            env,
        )


def read_yaml(path: Path) -> dict:
    import yaml

    require_file(path)
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}


def write_yaml(path: Path, content: dict) -> None:
    import yaml

    path.write_text(
        yaml.safe_dump(content, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print(f"Конфиг сформирован: {path.name}")


def infer_categorical_features(train_path: str, controls: set[str]) -> list[str]:
    import pyarrow.dataset as ds
    import pyarrow.types as types

    schema = ds.dataset(train_path, format="parquet").schema
    return sorted(
        field.name
        for field in schema
        if field.name not in controls
        and (
            types.is_string(field.type)
            or types.is_large_string(field.type)
            or types.is_dictionary(field.type)
            or types.is_boolean(field.type)
        )
    )


def ensure_selector_data_config() -> None:
    if SELECTOR_DATA_CONFIG.is_file():
        return

    require_file(SPLIT_MANIFEST)
    manifest = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
    paths = manifest.get("paths", {})
    if not paths.get("train") or not paths.get("valid"):
        raise ValueError("split_manifest.json не содержит путей train/valid")

    sample = read_yaml(SAMPLE_CONFIG) if SAMPLE_CONFIG.is_file() else {}
    sample_input = sample.get("input", {})
    sample_split = sample.get("split", {})
    previous_data = {}
    if INTERACTION_CONFIG.is_file():
        previous_data = read_yaml(INTERACTION_CONFIG).get("data", {})

    group_columns = sample_split.get("group_columns") or []
    leakage_keys = list(dict.fromkeys([*LEAKAGE_KEY_COLUMNS, *group_columns]))
    controls = {
        TARGET,
        *ID_COLUMNS,
        *SAMPLING_KEY_COLUMNS,
        *leakage_keys,
        *BOOTSTRAP_KEY_COLUMNS,
        *EXCLUDED_FEATURES,
    }
    categorical_features = (
        CATEGORICAL_FEATURES
        or sample_input.get("categorical_features")
        or previous_data.get("categorical_features")
        or infer_categorical_features(paths["train"], controls)
    )

    data = {
        "train_path": paths["train"],
        "valid_path": paths["valid"],
        "target": TARGET,
        "positive_label": POSITIVE_LABEL,
        "id_columns": ID_COLUMNS,
        "sampling_key_columns": SAMPLING_KEY_COLUMNS,
        "leakage_key_columns": leakage_keys,
        "bootstrap_key_columns": BOOTSTRAP_KEY_COLUMNS,
        "categorical_features": categorical_features,
        "categorical_null_strategy": CATEGORICAL_NULL_STRATEGY,
        "categorical_null_value": CATEGORICAL_NULL_VALUE,
        "excluded_features": EXCLUDED_FEATURES,
        "required_features": REQUIRED_FEATURES,
    }
    for role in ("oos", "oot", "test"):
        if paths.get(role):
            data[f"{role}_path"] = paths[role]

    SELECTOR_DATA_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    write_yaml(SELECTOR_DATA_CONFIG, {"data": data})
    print("selector_data.yaml восстановлен из существующих split-файлов.")


def build_sample_config() -> None:
    config = read_yaml(SAMPLE_TEMPLATE)
    config["input"].update(
        {
            "path": str(SOURCE_PATH),
            "format": SOURCE_FORMAT,
            "target": TARGET,
            "positive_label": POSITIVE_LABEL,
            "id_columns": ID_COLUMNS,
            "sampling_key_columns": SAMPLING_KEY_COLUMNS,
            "leakage_key_columns": LEAKAGE_KEY_COLUMNS,
            "bootstrap_key_columns": BOOTSTRAP_KEY_COLUMNS,
            "categorical_features": CATEGORICAL_FEATURES,
            "categorical_null_strategy": CATEGORICAL_NULL_STRATEGY,
            "categorical_null_value": CATEGORICAL_NULL_VALUE,
            "excluded_features": EXCLUDED_FEATURES,
            "required_features": REQUIRED_FEATURES,
        }
    )
    config["split"].update(
        {
            "mode": SPLIT_MODE,
            "train_fraction": TRAIN_FRACTION,
            "valid_fraction": VALID_FRACTION,
            "oos_fraction": OOS_FRACTION,
            "oot_fraction": OOT_FRACTION,
            "test_fraction": TEST_FRACTION,
            "time_column": TIME_COLUMN,
            "group_columns": GROUP_COLUMNS,
        }
    )
    config["output"].update(
        {"directory": str(SPLIT_DIR), "overwrite": OVERWRITE_SPLITS}
    )
    config["folds"].update(
        {"enabled": RUN_MODE == "stability", "n_folds": N_FOLDS}
    )
    write_yaml(SAMPLE_CONFIG, config)


def build_interaction_config() -> None:
    data = read_yaml(SELECTOR_DATA_CONFIG)["data"]
    data.update(
        {
            "target": TARGET,
            "positive_label": POSITIVE_LABEL,
            "id_columns": ID_COLUMNS,
            "sampling_key_columns": SAMPLING_KEY_COLUMNS,
            "leakage_key_columns": LEAKAGE_KEY_COLUMNS,
            "bootstrap_key_columns": BOOTSTRAP_KEY_COLUMNS,
            "categorical_null_strategy": CATEGORICAL_NULL_STRATEGY,
            "categorical_null_value": CATEGORICAL_NULL_VALUE,
            "excluded_features": EXCLUDED_FEATURES,
            "required_features": REQUIRED_FEATURES,
        }
    )
    config = read_yaml(INTERACTION_TEMPLATE)
    config["data"] = data
    config["search"].update(
        {
            "primary_metric": PRIMARY_METRIC,
            "min_features": MIN_FEATURES,
            "max_features": MAX_FEATURES,
            "max_gap": MAX_GAP,
        }
    )
    has_oos = bool(data.get("oos_path"))
    config["robust_validation"].update(
        {
            "require_oos": has_oos,
            "evaluate_oot": bool(data.get("oot_path")),
            "evaluate_test": bool(data.get("test_path")),
            "bootstrap_repeats": BOOTSTRAP_REPEATS,
        }
    )
    config["decision_threshold"].update(
        {
            "min_recall": MIN_RECALL,
            "min_precision": MIN_PRECISION,
            "require_oos_feasible": has_oos,
        }
    )
    config["execution"].update(
        {
            "backend": "ray" if INSTALL_RAY else "local",
            "parallel_trials": 0,
            "threads_per_trial": 0,
            "gpu_devices": [] if INSTALL_RAY else GPU_DEVICES,
        }
    )
    config["resources"].update(
        {
            "mode": "hybrid",
            "cpu_cores": CPU_CORES,
            "ram_gb": RAM_GB,
            "gpu_memory_by_device_gb": GPU_MEMORY_BY_DEVICE_GB,
            "reserve_gpu_memory_gb": GPU_RESERVE_MEMORY_GB,
            "max_gpu_memory_utilization": GPU_MAX_MEMORY_UTILIZATION,
            "calibration_safety_factor": CALIBRATION_SAFETY_FACTOR,
        }
    )
    config["model_params"].update(
        {"task_type": TASK_TYPE, "gpu_ram_part": GPU_RAM_PART}
    )
    config["output"].update(
        {
            "directory": str(DIRECT_OUTPUT),
            "cache_directory": str(DIRECT_OUTPUT / "cache"),
        }
    )
    write_yaml(INTERACTION_CONFIG, config)


def build_stability_config() -> None:
    config = read_yaml(STABILITY_TEMPLATE)
    config.update(
        {
            "selector_script": str(SELECTOR_SCRIPT),
            "selector_template_config": str(INTERACTION_CONFIG),
            "selector_data_config": str(SELECTOR_DATA_CONFIG),
            "folds_manifest": str(FOLDS_MANIFEST),
            "output_directory": str(STABILITY_OUTPUT),
            "run_final": RUN_FINAL_SELECTOR,
        }
    )
    write_yaml(STABILITY_CONFIG, config)


In [ ]:
# 3. ПРОВЕРКА ФАЙЛОВ И УСТАНОВКА ЗАВИСИМОСТЕЙ
if RUN_MODE not in {"direct", "stability", "none"}:
    raise ValueError("RUN_MODE должен быть direct, stability или none")

for path in (
    INTERACTION_TEMPLATE,
    STABILITY_TEMPLATE,
    SELECTOR_SCRIPT,
    STABILITY_RUNNER,
):
    require_file(path)

if RUN_SAMPLE_PREPARATION:
    require_file(SAMPLE_TEMPLATE)
    require_file(PREPARE_SCRIPT)

if INSTALL_REQUIREMENTS:
    install_packages()
else:
    print("Установка зависимостей пропущена.")


In [ ]:
# 4. ПОДГОТОВКА TRAIN / VALID / OOS / OOT / TEST И ФОЛДОВ
if RUN_SAMPLE_PREPARATION:
    if not SOURCE_PATH.exists():
        raise FileNotFoundError(f"Не найден SOURCE_PATH: {SOURCE_PATH}")
    build_sample_config()
    run([
        sys.executable,
        str(PREPARE_SCRIPT),
        "--config",
        str(SAMPLE_CONFIG),
    ])
else:
    print(
        "Подготовка выборок пропущена: существующие split_parts и "
        "config.sample-preprocessing.yaml не изменяются."
    )


In [ ]:
# 5. ФОРМИРОВАНИЕ АКТУАЛЬНЫХ INTERACTION / STABILITY КОНФИГОВ
# Если старый прогон не создал selector_data.yaml, он будет восстановлен
# из split_manifest.json, рабочего sample-конфига и схемы train Parquet.
ensure_selector_data_config()
build_interaction_config()

if RUN_MODE == "stability":
    require_file(FOLDS_MANIFEST)
    build_stability_config()


In [ ]:
# 6. ЗАПУСК ОТБОРА ФАКТОРОВ
if RUN_MODE == "direct":
    run([
        sys.executable,
        str(SELECTOR_SCRIPT),
        "--config",
        str(INTERACTION_CONFIG),
    ])
elif RUN_MODE == "stability":
    run([
        sys.executable,
        str(STABILITY_RUNNER),
        "--config",
        str(STABILITY_CONFIG),
    ])
else:
    print("RUN_MODE=none: конфиги сформированы, selector не запускался.")
